# 07 Files, errors and modules

Reading and writing files, handling errors, and using code from other modules.

## Paths with pathlib

`pathlib.Path` handles file paths on Windows, Mac and Linux the same way.

In [1]:
from pathlib import Path

data_dir = Path("..") / "data"
print(data_dir.resolve())
print(sorted(p.name for p in data_dir.glob("*.csv")))

f = data_dir / "patients.csv"
print(f.exists(), f.suffix, f.stem)

/home/claude/python-data-foundations/data
['isolates_raw.csv', 'patients.csv', 'visits.csv']
True .csv patients


## Reading and writing text files

`with open(...)` closes the file for you, even if something goes wrong.
`"r"` reads, `"w"` writes (replacing the file), `"a"` appends.

In [2]:
out = Path("notes.txt")

with open(out, "w") as fh:
    fh.write("first line\n")
    fh.write("second line\n")

with open(out) as fh:
    for line in fh:
        print(line.strip())

out.unlink()       # delete the demo file

first line
second line


## CSV files with the csv module

For quick jobs. For real analysis use pandas (notebook 11).

In [3]:
import csv

with open(data_dir / "patients.csv") as fh:
    reader = csv.DictReader(fh)
    first_rows = [next(reader) for _ in range(3)]

for row in first_rows:
    print(row["patient_id"], row["site"], row["age"])     # note: age is text here

P0001 Kampala 30
P0002 Mbarara 81
P0003 Gulu 24


## JSON

JSON maps directly onto Python dicts and lists.

In [4]:
import json

record = {"id": "P0012", "tests": {"hb": 10.4, "wbc": 6.1}, "flags": ["low_hb"]}

text = json.dumps(record, indent=2)
print(text)

back = json.loads(text)
print(back["tests"]["hb"])

# to a file:  json.dump(record, open("x.json", "w"))
# from a file: json.load(open("x.json"))

{
  "id": "P0012",
  "tests": {
    "hb": 10.4,
    "wbc": 6.1
  },
  "flags": [
    "low_hb"
  ]
}
10.4


## Errors and exceptions

When something goes wrong, Python raises an exception. Read the last line of the
error first: it names the type and the cause.

Common ones: `ValueError`, `TypeError`, `KeyError`, `IndexError`,
`FileNotFoundError`, `ZeroDivisionError`.

In [5]:
def to_number(text):
    try:
        return float(text)
    except ValueError:
        return None

print([to_number(x) for x in ["12.5", "n/a", "7", ""]])

[12.5, None, 7.0, None]


Catch the specific error you expect. A bare `except:` hides real bugs.
`else` runs when nothing failed, `finally` always runs.

In [6]:
try:
    with open(data_dir / "missing.csv") as fh:
        content = fh.read()
except FileNotFoundError as e:
    print("Could not open:", e.filename)
else:
    print("read", len(content), "characters")
finally:
    print("done")

Could not open: ../data/missing.csv
done


Raise your own errors when input does not make sense:

In [7]:
def bmi(weight_kg, height_cm):
    if height_cm <= 0:
        raise ValueError(f"height must be positive, got {height_cm}")
    return weight_kg / (height_cm / 100) ** 2

try:
    bmi(70, 0)
except ValueError as e:
    print(e)

height must be positive, got 0


## Modules and imports

A module is a `.py` file. Import a whole module, a part of it, or give it a short name.

In [8]:
import math
from statistics import mean, median
import datetime as dt

print(math.sqrt(16), mean([1, 2, 3]), median([3, 1, 2]))
print(dt.date.today())

4.0 2 2
2026-11-23


Your own functions can live in a file such as `helpers.py` next to the notebook,
then `from helpers import bmi`.

Install packages from a terminal:

```
pip install pandas seaborn
```

or with conda: `conda install pandas seaborn`.

## Dates with datetime

In [9]:
from datetime import date, datetime, timedelta

collected = datetime.strptime("14/03/2025", "%d/%m/%Y")
print(collected)
print(collected.strftime("%Y-%m-%d"))
print(collected + timedelta(days=28))
print((date(2025, 12, 31) - collected.date()).days, "days")

2025-03-14 00:00:00
2025-03-14
2025-04-11 00:00:00
292 days
